# 第9回 体験ワーク② ── 通信を「安全に使う」しくみ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

千葉大学 情報リテラシ（工学部1年）。第8回でやり残した演習の続きです。
パケットは色々な機器を通って旅します。途中で**のぞき見**されても困らないように、
私たちは通信を**暗号化**しています。そのしくみを手を動かして体験します。

## このノートでやる5つのワーク
1. **ワーク1：シーザー暗号** ── 鍵で暗号化・復号／鍵を知らないと総当たり（古典暗号の限界）
2. **ワーク2：ハッシュでパスワードを守る** ── なぜサービスは「パスワードそのもの」を保存しないのか
3. **ワーク3：平文 vs HTTPS** ── 🌐 証明書を覗いて、なぜ野良Wi-Fiが危険かを実感する
4. **ワーク4：公開鍵暗号のミニ体験** ── 「公開鍵で施錠・秘密鍵だけで開錠」を“おもちゃRSA”で体験する
5. **ワーク5：メールヘッダを読む** ── 「From」は信用できる？ 中継経路と SPF/DKIM/DMARC で詐称を見抜く

> 🌐 と書いたセルは**インターネット接続が必要**です（Colab では最初からつながっています）。


---
## ワーク1：シーザー暗号（古典暗号・換字式）

文字を決まった数だけ後ろにずらす、最も古い暗号の一つです。「ずらす数」が**鍵**。
例：鍵=3 なら A→D, B→E, ... 受け取った人は同じ鍵で逆向きにずらせば元に戻せます（**共通鍵**の考え方）。


In [ ]:
def caesar(text, shift):
    """text を shift だけずらす（英字のみ／その他はそのまま）"""
    out = []
    for ch in text:
        if "A" <= ch <= "Z":
            out.append(chr((ord(ch) - ord("A") + shift) % 26 + ord("A")))
        elif "a" <= ch <= "z":
            out.append(chr((ord(ch) - ord("a") + shift) % 26 + ord("a")))
        else:
            out.append(ch)
    return "".join(out)

plain = "Meet at noon"
key = 3
enc = caesar(plain, key)
dec = caesar(enc, -key)   # 逆向きにずらせば復号

print("平文       :", plain)
print(f"暗号文(鍵{key}):", enc)
print("復号       :", dec)
print("元に戻った? :", dec == plain)

### 鍵を知らないと？ ── 総当たり（ブルートフォース）
シーザー暗号は鍵が**25通りしかない**ので、全部試せばすぐ読めてしまいます。
だから現代の通信ではこんな単純な暗号は使いません（後のワーク3でその「本物」を見ます）。


In [ ]:
secret = caesar("Attack at dawn", 7)   # 鍵7で暗号化された文（鍵は秘密という設定）
print("受け取った暗号文:", secret)
print("--- 鍵を1〜25まで総当たり ---")
for k in range(1, 26):
    print(f"鍵{k:>2} : {caesar(secret, -k)}")
print("→ 意味の通る行が、正しい鍵です（人間ならすぐ分かる ＝ 弱い暗号）")

---
## ワーク2：ハッシュでパスワードを守る

まともなサービスは、あなたのパスワードを**そのままは保存しません**。
代わりに **ハッシュ**（一方向の変換）にかけた結果だけを保存します。

ハッシュの性質：
- 同じ入力 → **必ず同じ結果**（だからログイン時に照合できる）
- 1文字でも変える → **結果が激変**（雪崩効果）
- 結果から元の文字列に**戻せない**（一方向）


In [ ]:
import hashlib

def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

print("同じ入力は同じ結果になる:")
print("  password123 ->", sha256("password123"))
print("  password123 ->", sha256("password123"))
print()
print("1文字違うだけで結果は激変する（雪崩効果）:")
print("  password123 ->", sha256("password123"))
print("  password124 ->", sha256("password124"))

### なぜ「そのまま保存」は危ないのか
もしサービスがパスワードを平文で保存していたら、漏れた瞬間に全員のパスワードが読めてしまいます。
ハッシュなら、漏れても元のパスワードは（簡単には）分かりません。
…ただし**よくあるパスワードは「答え一覧表」で照合されると破られる**ので、複雑で長いものにすることが大事です。


In [ ]:
import hashlib

def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# サービスはこの「ハッシュ値」だけを保存していると考える
stored = sha256("chiba2026!")

# 攻撃者が「よくあるパスワード一覧」で総当たりするイメージ
guesses = ["123456", "password", "qwerty", "chiba", "chiba2026!", "iloveyou"]
print("保存されているハッシュ:", stored)
print("--- よくあるパスワードで照合 ---")
for g in guesses:
    hit = "★一致！破られた" if sha256(g) == stored else "不一致"
    print(f"  {g:<12} -> {hit}")
print("\n教訓: 短い・ありがちなパスワードは一覧照合で破られる。長く複雑にしよう。")

---
## ワーク3：平文 vs HTTPS ── なぜ野良Wi-Fiが危険か

`http://`（平文）は、はがきと同じで**通り道の誰でも中身を読めます**。
`https://` は封筒に入れて鍵をかけた状態。途中の機器には**暗号化された中身**しか見えません。

公衆・野良Wi-Fiでは「通り道」に悪意ある人がいるかもしれない。
だから **`https://`（鍵マーク）になっているか**が大事なのです。まずは本物の証明書を覗いてみましょう。


In [ ]:
# 🌐 要ネット：HTTPSサーバの「証明書」を覗く（封筒に貼られた本人確認シール）
import ssl, socket

host = "www.chiba-u.ac.jp"   # 👈 好きなサイトに変えてOK
ctx = ssl.create_default_context()
with ctx.wrap_socket(socket.socket(), server_hostname=host) as s:
    s.settimeout(10)
    s.connect((host, 443))
    cert = s.getpeercert()

subject = dict(x[0] for x in cert["subject"])
issuer = dict(x[0] for x in cert["issuer"])
print("接続先          :", host)
print("証明書の持ち主   :", subject.get("commonName", "(不明)"))
print("発行した認証局   :", issuer.get("organizationName", issuer.get("commonName", "(不明)")))
print("有効期限         :", cert.get("notAfter"))
print("\n→ 鍵マークの裏では、信頼できる第三者（認証局）が『この相手は本物』と保証している。")

### 🌐 http と https で「中身の見え方」を比べる
同じ内容でも、平文(http)は通り道に丸見え、https は暗号化されて見えません。
ここでは「httpsで取った通信は中身が暗号化されている」ことを、接続できる事実として確認します。


In [ ]:
# 🌐 要ネット：https でアクセスして、暗号化された接続が確立できることを見る
import urllib.request, ssl

url = "https://example.com"
req = urllib.request.Request(url, headers={"User-Agent": "chiba-infolit"})
with urllib.request.urlopen(req, timeout=10) as r:
    print("URL        :", url)
    print("ステータス :", r.status, "(200 ならOK)")
    print("サーバ     :", r.headers.get("Server"))
    print("中身の先頭 :", r.read(80).decode("utf-8", "ignore").replace("\n", " "), "...")
print("\n→ この通信路は暗号化済み。野良Wi-Fiでも、httpsなら中身は途中で読まれない。")
print("  逆に http:// だと、同じ内容が通り道に丸見えになる（だから鍵マークを確認！）")

---
## ワーク4：公開鍵暗号のミニ体験（封筒に鍵）

ワーク1のシーザー暗号は**共通鍵**でした。送る人も受け取る人も「同じ鍵」を持っていないと使えません。
でも、これには大問題があります。**その鍵を、どうやって相手に安全に渡す？** 鍵を送る途中でのぞき見されたら、もうおしまいです。

そこで登場するのが **公開鍵暗号**。鍵を2つに分けるのがミソです。

- **公開鍵**（みんなに配ってOK）… これは「**施錠専用**」の鍵。誰でも鍵をかけられる。
- **秘密鍵**（自分だけが持つ）… これは「**開錠専用**」の鍵。これがないと開けられない。

（あとで出てくる数で言うと、**公開鍵 = (e, n) のペア**、**秘密鍵 = d** です。e や n はみんなに配ってよく、d だけは絶対に自分の中にしまっておきます。）

イメージは「**南京錠つきの封筒**」。施錠する錠前（公開鍵）は誰にでも配れます。
でも、その錠前を開けられる鍵（秘密鍵）は受け手しか持っていない。
だから**錠前を盗み見られても、中身は受け手しか取り出せない**のです。

> 第9回でやる **TLSハンドシェイク** はこれを使います。
> 「**最初だけ**公開鍵暗号で共通鍵を安全に手渡し → **本番は速い共通鍵**で大量データをやりとり」。
> この合わせ技を **ハイブリッド暗号** と呼びます（公開鍵は安全だが遅い、共通鍵は速いので、いいとこ取り）。
>
> ※ 最新のTLS（TLS 1.3）では、共通鍵の作り方にここで見るRSAとは別の方式（鍵交換）も使われますが、
> 「**最初だけ手間をかけて共通鍵を安全に用意し、本番は速い共通鍵で大量にやりとりする**」という考え方は同じです。

In [ ]:
# おもちゃRSA：公開鍵暗号のしくみを、わざと小さな数で体験する
# （本物は数百桁の巨大な素数を使うので、現実には逆算できない。ここは原理の確認用）

def egcd(a, b):
    """拡張ユークリッドの互除法（最大公約数と係数を求める）"""
    if b == 0:
        return (a, 1, 0)
    g, x, y = egcd(b, a % b)
    return (g, y, x - (a // b) * y)

def modinv(a, m):
    """a の『mを法とした逆数』＝ a*d ≡ 1 (mod m) となる d を求める"""
    g, x, _ = egcd(a % m, m)
    return x % m

# --- 1) 鍵を作る ---
p, q = 17, 23            # 2つの素数（本物は超巨大な素数）
n = p * q                # n=積。公開鍵の一部（公開鍵は (e, n) のペア）。素因数分解で破れるが巨大だと無理
phi = (p - 1) * (q - 1)  # φ(n)：鍵の計算に使う秘密の数
e = 7                    # 公開指数（公開鍵）。φと互いに素な数を選ぶ
d = modinv(e, phi)       # 秘密指数（秘密鍵）。e の φ に対する逆元

print("=== 鍵の中身 ===")
print(f"  p, q = {p}, {q}   ← 秘密の素数2つ")
print(f"  n = p*q = {n}     ← 公開鍵の一部（公開鍵は (e, n) のペア。みんなに配る）")
print(f"  φ = (p-1)(q-1) = {phi}  ← 秘密の数")
print(f"  e = {e}           ← 公開指数（公開鍵）= 施錠用")
print(f"  d = {d}          ← 秘密指数（秘密鍵）= 開錠用  ※ e*d ≡ 1 (mod φ) を満たす")
print(f"  確認: e*d % φ = {e*d % phi}  （1ならOK）")

# --- 2) 暗号化と復号 ---
m = 42                   # 送りたい秘密の数（メッセージ。n より小さいこと）
c = pow(m, e, n)         # 施錠：公開鍵 e で暗号化（= (m**e) % n。pow の3引数版が同じ計算）
m2 = pow(c, d, n)        # 開錠：秘密鍵 d で復号 （= (c**d) % n）

print("\n=== 暗号化 → 復号 ===")
print(f"  平文 m            = {m}")
print(f"  暗号文 c = m^e %n = {c}   ← 公開鍵で施錠。これを盗み見られても…")
print(f"  復号 m' = c^d %n  = {m2}   ← 秘密鍵を持つ人だけが元に戻せる")
print(f"  元に戻った?       = {m == m2}")

# --- 3) おまけ：署名は「逆向き」 ---
# 暗号化は「公開鍵で施錠 → 秘密鍵で開錠」。署名はその逆で「秘密鍵で施錠 → 公開鍵で確認」。
# 本人しか持たない秘密鍵で作れる＝「確かに本人が書いた」という証明になる。
sign = pow(m, d, n)      # 署名：秘密鍵 d で作る（本人だけが作れる）
verify = pow(sign, e, n) # 検証：公開鍵 e で確認（誰でもチェックできる）

print("\n=== 署名（逆向きの使い方）===")
print(f"  署名 = m^d %n = {sign}   ← 秘密鍵で作る（本人にしか作れない）")
print(f"  検証 = 署名^e %n = {verify}   ← 公開鍵で確認。元の m と一致すれば本物")
print(f"  本人の署名だと確認できた? = {verify == m}")

### なぜこれで「鍵の配り方」問題が解けるのか

公開鍵（施錠用）は**世界中に配ってOK**です。盗み見られても、それでできるのは「施錠」だけ。
開けられるのは**秘密鍵を持つ受け手だけ**。

つまり——
- あなたは相手の**公開鍵で共通鍵を施錠**して送る
- 途中の機器や攻撃者には**暗号化された中身**しか見えない
- 相手は**自分の秘密鍵で開錠**して、共通鍵を取り出す

こうして「**最初の共通鍵を安全に渡す**」ことができます。
あとはその共通鍵（速い）で本番のやりとりをすればいい——これが第9回の**TLS／ハイブリッド暗号**の心臓部です。

---
## ワーク5：メールヘッダを読む（Fromは信用できる？）

迷惑メールやフィッシング詐欺は、差出人(**From**)を平気で偽ります。
じつはメールの **From は「自己申告」** に過ぎません。手紙の封筒に自分で好きな差出人名を書くのと同じで、
**書こうと思えば誰でも好きな名前を書けてしまう**のです。

では何を見れば手がかりになるのか。メールの**ヘッダ**には、本文には出てこない情報が詰まっています。

- **Received 行**＝メールが通ってきた**中継サーバの足跡**。
  ふつう複数あり、**下にある行ほど古い（最初の送信元に近い）／上にある行ほど新しい（あなたに近い）**。
  下から上へ読むと「どこから出発して、どんな経路で届いたか」が分かります。
- **Authentication-Results 行**＝受信側サーバが行った**なりすまし検査の結果**。
  - **SPF**：その送信サーバ（IP）から、名乗っているドメインのメールを送ってよいか（送信元のお墨付き）
  - **DKIM**：送信側が付けた電子署名で、本文やヘッダが**途中で改ざんされていないか**を確認するしくみ
  - **DMARC**：SPF/DKIM の結果が合わなかったメールをどう扱うか（受け取る/捨てる等）を決めるドメイン側のルール
  - これらが **pass** なら詐称の可能性は下がり、**fail** なら要注意のサインです。

In [ ]:
# 標準ライブラリ email だけでメールヘッダを解析する（pip不要）
import re
from email.parser import Parser

# サンプルの生メールヘッダ（実際のメールの「ソース表示」に近いダミー）
raw_header = """Received: from mx.chiba-u.ac.jp (mx.chiba-u.ac.jp [133.82.1.10])
\tby mail.example.net with ESMTPS id ABC123
\tfor <you@example.net>; Tue, 09 Jun 2026 10:32:11 +0900
Received: from sender.example.com (sender.example.com [203.0.113.55])
\tby mx.chiba-u.ac.jp with ESMTP id XYZ789;
\tTue, 09 Jun 2026 10:31:58 +0900
Authentication-Results: mx.chiba-u.ac.jp;
\tspf=pass smtp.mailfrom=info@sender.example.com;
\tdkim=pass header.d=sender.example.com;
\tdmarc=pass header.from=sender.example.com
From: "Campus Notice" <info@sender.example.com>
To: you@example.net
Subject: [important] account confirmation
Date: Tue, 09 Jun 2026 10:31:55 +0900
Message-ID: <20260609103155.AB12@sender.example.com>

(本文は省略)
"""

msg = Parser().parsestr(raw_header)

# --- 1) 基本の差出人情報（※ From は自己申告であることに注意）---
print("=== 基本情報（From は自己申告！）===")
print("  From       :", msg.get("From"))
print("  To         :", msg.get("To"))
print("  Subject    :", msg.get("Subject"))
print("  Date       :", msg.get("Date"))
print("  Message-ID :", msg.get("Message-ID"))

# --- 2) Received を「古い順」に並べて経路を表示 ---
# ヘッダでは新しい順（上が新しい）に積まれているので、逆順にすると古い順＝出発地→到着地
received = msg.get_all("Received") or []
print("\n=== 中継経路（古い順＝出発地 → あなた）===")
for i, line in enumerate(reversed(received), start=1):
    one_line = " ".join(line.split())   # 折り返しを1行にまとめる
    print(f"  [{i}] {one_line}")

# --- 3) なりすまし検査の結果（SPF / DKIM / DMARC）---
ar = msg.get("Authentication-Results") or ""
print("\n=== なりすまし検査（Authentication-Results）===")
for check in ["spf", "dkim", "dmarc"]:
    m = re.search(check + r"=(\w+)", ar)
    result = m.group(1) if m else "(なし)"
    mark = "✅ OK" if result == "pass" else ("⚠️ 要注意" if result == "fail" else "❓")
    print(f"  {check.upper():<6}: {result:<6} {mark}")

print("\n→ このメールは spf/dkim/dmarc が pass。詐称の可能性は低い。")
print("  もし spf=fail や dkim=fail なら、From が本物でも疑ってかかること！")

### 自分のメールで試してみよう（本物のヘッダに貼り替え）

上の `raw_header` は練習用のダミーです。**本物のヘッダ**でも同じことができます。

**Gmail の場合**：開いたメールの右上「︙(その他)」→「**メッセージのソースを表示**」を開くと、生のヘッダが全部見られます。
それをコピーして、上のコードの `raw_header` の中身に**貼り替えて**実行してみましょう。

**From詐称・フィッシングの見抜き方（チェックリスト）**
- **From のドメイン**が、本当にその組織のものか？（`chiba-u.ac.jp` を装った `chiba-u.example.com` などに注意）
- **spf / dkim / dmarc** が pass か？ fail があれば強く疑う。
- **Received の一番古い行（出発地）**が、差出人として自然な場所か？ 見覚えのない海外サーバ等は赤信号。
- 文面が「今すぐ」「アカウント停止」など**急かす**／**リンクをクリックさせたがる**ものは特に警戒。

> 「From は封筒の自己申告、Received は消えない足跡、SPF/DKIM/DMARC は受付の本人確認」。
> この3点をセットで見れば、なりすましにだまされにくくなります。

---
## ふりかえり（提出は不要・考えてみよう）
1. シーザー暗号の鍵は25通り。もし「26文字を好きに並べ替える」換字式なら鍵は何通り？（ヒント：26の階乗）
2. 自分がよく使うパスワードの長さ・種類は、ワーク2の「一覧照合」で破られそう？
3. スマホで野良Wi-Fiに繋いだとき、まず何を確認すればいい？（鍵マーク／`https`）
4. 公開鍵暗号で「**公開鍵を盗み見られても安全**」なのはなぜ？ 施錠用と開錠用が別、という言葉で説明してみよう。
5. 怪しいメールが届いたら、**From 以外**にどこを見れば本物か見分けられる？（Received／SPF・DKIM・DMARC）

> 第8回「旅」の地図：**住所(IP)→電話帳(DNS)→同じ町か(サブネット)→封筒に鍵(HTTPS)**。
> 2冊のノートで、その旅を最初から最後まで自分の手で体験しました。おつかれさまでした！
